# FinLLM GPU Training (Google Colab)

This notebook runs the **exact same CLI** (`main.py`) already tested on CPU in this project - it does not reimplement anything. Every command below is a real command from this repo's tested `main.py`; nothing here is aspirational.

**Before running:** Runtime -> Change runtime type -> Hardware accelerator -> GPU (T4 is free-tier).

**What this notebook actually does, in order:**
1. Verify the GPU is real (not assumed)
2. Get the repository onto the Colab VM
3. Install dependencies
4. Run the same smoke tests already passed on CPU, now on GPU
5. Prepare data with a larger token budget than was practical on CPU
6. Run the `small` or `financial_poc` preset (previously calculated at ~300h / ~2400h on this project's CPU - genuinely impractical there, genuinely practical on a T4)
7. Save checkpoints to Google Drive so they survive the Colab session ending
8. Run evaluation and print real, measured results

No step here is skipped or faked - if a cell fails, later cells will fail honestly rather than silently produce placeholder output.

## 1. Verify the GPU is real

In [ ]:
!nvidia-smi
import torch
print('cuda_available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
else:
    raise RuntimeError('No GPU detected. Set Runtime -> Change runtime type -> GPU before continuing.')

## 2. Get the repository
Two options - use whichever matches how you got this notebook onto Colab:

In [ ]:
# OPTION A: clone from your own GitHub remote (edit the URL first)
# !git clone https://github.com/<your-username>/DeepSeek-from-Scratch.git
# %cd DeepSeek-from-Scratch

# OPTION B: upload the repo as a zip via the Colab file browser, then:
# !unzip -q DeepSeek-from-Scratch.zip
# %cd DeepSeek-from-Scratch

print('Uncomment and run ONE of the options above, matching how you brought the repo here.')

## 3. Install dependencies (only what's missing/compatible - do not blindly upgrade Colab's preinstalled torch)

In [ ]:
!pip install -q tiktoken datasets pyyaml pypdf python-docx scikit-learn networkx langdetect
# Colab already ships a CUDA-enabled torch - do not reinstall the CPU wheel from requirements.txt.
!python -c "import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())"

## 4. Mount Google Drive for persistent checkpoint storage
Colab sessions are ephemeral - verify Drive is actually mounted before trusting it, per this project's own rule against claiming persistence that wasn't verified.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/FinLLM'
for sub in ('checkpoints', 'experiments', 'logs', 'datasets', 'artifacts'):
    os.makedirs(f'{DRIVE_ROOT}/{sub}', exist_ok=True)

assert os.path.isdir(DRIVE_ROOT), 'Drive did not actually mount - do not proceed assuming it did.'
print('Verified Drive mounted at', DRIVE_ROOT)

# Symlink the repo's checkpoints/ dir to Drive so `main.py train` writes
# there directly with zero code changes.
!rm -rf checkpoints && ln -s {DRIVE_ROOT}/checkpoints checkpoints
!rm -rf experiments && ln -s {DRIVE_ROOT}/experiments experiments

## 5. Smoke tests - the same ones already passed on CPU, now verifying the GPU path

In [ ]:
!python main.py runtime
!python main.py demo

## 6. Prepare data
On CPU, `tiny_debug` (1M tokens) was the only preset actually run - `small` (10M) and `financial_poc` (50M) were explicitly NOT run because the CPU time estimate was ~300h / ~2400h. On a T4 this becomes practical; increase `--max-tokens` per source as GPU throughput allows.

In [ ]:
!python main.py dataset list
!python main.py dataset prepare --name fineweb_edu --max-tokens 10000000
!python main.py dataset prepare --name financial_text_investopedia --max-tokens 3000000
!python main.py dataset prepare --name financial_qa_sujet --max-tokens 3000000
!python main.py dataset prepare --name financial_reasoning_finqa --max-tokens 1000000
!python main.py dataset prepare --name financial_reports_edgar --max-tokens 3000000
!python main.py dataset prepare --name financial_instruction_alpaca --max-tokens 1000000
!python main.py dataset dashboard --preset small

## 7. Train - Stage A (base pretraining)

In [ ]:
!python main.py train --preset small

## 8. Train - Stage B (financial instruction tuning)
Point `--checkpoint` at whatever Stage A actually produced (check the printed path from the previous cell).

In [ ]:
!ls checkpoints/base/*.pt
# Edit the checkpoint filename below to match Stage A's actual output:
!python main.py train --stage instruction --checkpoint checkpoints/base/checkpoint_2000.pt --max-steps 500

## 9. Evaluate - real measured results, not asserted ones

In [ ]:
!ls checkpoints/instruction/*.pt
# Edit the checkpoint filename below to match Stage B's actual output:
!python main.py evaluate --checkpoint checkpoints/instruction/checkpoint_500.pt --output /content/drive/MyDrive/FinLLM/experiments/eval_gpu_run.json

## 10. Compare against the CPU-trained checkpoints (optional)
If you copy the CPU-trained `checkpoints/base/checkpoint_100.pt` / `checkpoints/instruction/checkpoint_100.pt` into this Colab session, you can directly compare the 100-step CPU pipeline-test run against a real GPU-scale run.

In [ ]:
# !python main.py compare --base checkpoints/base/checkpoint_100.pt --instruction checkpoints/instruction/checkpoint_500.pt --output /content/drive/MyDrive/FinLLM/experiments/compare_gpu_vs_cpu.json

---
### After this notebook runs
Copy the resulting `checkpoints/base/*.json` and `checkpoints/instruction/*.json` metadata files (not the large `.pt` binaries) back into the local repo's `checkpoints/` folders, and update `ai_platform/registry.py` / the final report with the real measured GPU training results - do not report numbers from this notebook without them actually having been produced by a run.